In [1]:
import tensorflow as tf
import numpy as np
import cv2
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_lfw_people
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping

# Load LFW dataset with fewer classes (min 50 faces per person)
lfw_people = fetch_lfw_people(min_faces_per_person=50, resize=0.5)
images = lfw_people.images
labels = lfw_people.target
label_names = lfw_people.target_names
num_classes = len(label_names)

print("Total samples:", images.shape[0])
print("Image shape:", images.shape[1:])
print("Number of classes:", num_classes)

# Resize and preprocess images for VGG16
resized_images = np.zeros((images.shape[0], 224, 224, 3), dtype='float32')
for i in range(images.shape[0]):
    img = cv2.cvtColor(images[i], cv2.COLOR_GRAY2RGB)
    resized_images[i] = cv2.resize(img, (224, 224))
resized_images = preprocess_input(resized_images)

# One-hot encode labels
labels_cat = to_categorical(labels, num_classes)

# Train-test split
X_train, X_test, y_train, y_test, labels_train, labels_test = train_test_split(
    resized_images, labels_cat, labels, test_size=0.2, random_state=42)

# Compute class weights to address imbalance
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(labels_train), y=labels_train)
class_weight_dict = dict(enumerate(class_weights))

# Image data augmentation
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

train_gen = datagen.flow(X_train, y_train, subset='training', batch_size=32)
val_gen = datagen.flow(X_train, y_train, subset='validation', batch_size=32)

# Load base VGG16 model
vgg_base = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
vgg_base.trainable = False

# Build model with fine-tuned VGG16
x = tf.keras.layers.GlobalAveragePooling2D()(vgg_base.output)
x = tf.keras.layers.Dense(256, activation='relu')(x)
x = tf.keras.layers.Dropout(0.5)(x)
output = tf.keras.layers.Dense(num_classes, activation='softmax')(x)

model = tf.keras.models.Model(inputs=vgg_base.input, outputs=output)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

# Early stopping
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Train model
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=20,
    class_weight=class_weight_dict,
    callbacks=[early_stop]
)

# Evaluate model
loss, accuracy = model.evaluate(X_test, y_test)
print("\n✅ Test Accuracy:", accuracy)

# Plot accuracy
plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Validation')
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training vs Validation Accuracy")
plt.legend()
plt.grid(True)
plt.show()



Total samples: 1560
Image shape: (62, 47)
Number of classes: 12



Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 224, 224, 3)]     0         
                                                                 
 block1_conv1 (Conv2D)       (None, 224, 224, 64)      1792      
                                                                 
 block1_conv2 (Conv2D)       (None, 224, 224, 64)      36928     
                                                                 
 block1_pool (MaxPooling2D)  (None, 112, 112, 64)      0         
                                                                 
 block2_conv1 (Conv2D)       (None, 112, 112, 128)     73856     
                                                                 
 block2_conv2 (Conv2D)       (None, 112, 112, 128)     147584    
                                                           

KeyboardInterrupt: 